In [0]:
%pip install optuna==4.8.0

In [0]:
%pip install xgboost==3.2.0

In [0]:
%restart_python

In [0]:
# Importing libraries 

import optuna
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import average_precision_score
import os

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

# Loading data

def load_numpy(train_table, test_table):
    train_df = spark.table(train_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = spark.table(test_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))

    def extract(df):
        arr = df.withColumn("features_arr", vector_to_array("features"))
        pdf = arr.select("features_arr", "is_fraud").toPandas()
        return (np.array(pdf["features_arr"].tolist()),
                pdf["is_fraud"].values)

    X_train, y_train = extract(train_df)
    X_test,  y_test  = extract(test_df)
    return X_train, y_train, X_test, y_test


print("Loading applications data...")
X_app_train, y_app_train, X_app_test, y_app_test = load_numpy(
    "workspace.ml_layer.application_train_features",
    "workspace.ml_layer.application_test_features"
)

print("Loading transactions data...")
X_txn_train, y_txn_train, X_txn_test, y_txn_test = load_numpy(
    "workspace.ml_layer.transaction_train_features",
    "workspace.ml_layer.transaction_test_features"
)

# Creating Optuna Objective Function

def objective_xgb(trial, X_train, y_train, X_val, y_val):
    """
    Optuna objective for XGBoost — optimizes PR-AUC.
    """
    fraud_count      = int(y_train.sum())
    legit_count      = len(y_train) - fraud_count
    scale_pos_weight = legit_count / fraud_count

    params = {
        "n_estimators"          : trial.suggest_int("n_estimators", 200, 600, step=50),
        "max_depth"             : trial.suggest_int("max_depth", 3, 8),
        "learning_rate"         : trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample"             : trial.suggest_float("subsample", 0.6, 0.95),
        "colsample_bytree"      : trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "colsample_bylevel"     : trial.suggest_float("colsample_bylevel", 0.5, 0.9),
        "min_child_weight"      : trial.suggest_int("min_child_weight", 5, 50),
        "gamma"                 : trial.suggest_float("gamma", 0.0, 0.5),
        "reg_alpha"             : trial.suggest_float("reg_alpha", 0.0, 0.5),
        "reg_lambda"            : trial.suggest_float("reg_lambda", 0.5, 3.0),
        "scale_pos_weight"      : scale_pos_weight,
        "eval_metric"           : "aucpr",
        "early_stopping_rounds" : 30,
        "random_state"          : 42,
        "n_jobs"                : -1,
        "verbosity"             : 0
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_proba = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, y_proba)  # PR-AUC


# Running Optuna Studies

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/hyperparameter_tuning")

N_TRIALS = 30  

# Applications XGBoost
print("\n" + "="*60)
print("  OPTUNA — XGBoost Applications")
print("="*60)

with mlflow.start_run(run_name="Optuna_XGB_applications"):
    study_app = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
    )
    study_app.optimize(
        lambda trial: objective_xgb(trial, X_app_train, y_app_train,
                                     X_app_test, y_app_test),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    print(f"\nBest PR-AUC: {study_app.best_value:.4f}")
    print(f"Best params:")
    for k, v in study_app.best_params.items():
        print(f"  {k}: {v}")

    mlflow.log_metric("best_pr_auc", study_app.best_value)
    mlflow.log_params(study_app.best_params)

#Transactions XGBoost 
print("\n" + "="*60)
print("  OPTUNA — XGBoost Transactions")
print("="*60)

# Stratified subsample transactions 

idx_fraud = np.where(y_txn_train == 1)[0]
idx_legit = np.where(y_txn_train == 0)[0]

# Keeping all frauds and reducing legit transactions to maintain the total 1 million records
n_fraud = len(idx_fraud)
n_legit_sample = 1_000_000 - n_fraud

# Choosing only legit records
idx_legit_sample = np.random.choice(idx_legit, size=n_legit_sample, replace=False)

# Shuffle
sample_idx = np.concatenate([idx_fraud, idx_legit_sample])
np.random.shuffle(sample_idx)

X_txn_train_sub  = X_txn_train[sample_idx]
y_txn_train_sub  = y_txn_train[sample_idx]

with mlflow.start_run(run_name="Optuna_XGB_transactions"):
    study_txn = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
    )
    study_txn.optimize(
        lambda trial: objective_xgb(trial, X_txn_train_sub, y_txn_train_sub,
                                     X_txn_test, y_txn_test),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    print(f"\nBest PR-AUC: {study_txn.best_value:.4f}")
    print(f"Best params:")
    for k, v in study_txn.best_params.items():
        print(f"  {k}: {v}")

    mlflow.log_metric("best_pr_auc", study_txn.best_value)
    mlflow.log_params(study_txn.best_params)

# Saving best parameters to Delta

best_params_records = []

for param_name, value in study_app.best_params.items():
    best_params_records.append({
        "model"      : "XGBoost",
        "dataset"    : "applications",
        "parameter"  : param_name,
        "value"      : float(value),
        "best_pr_auc": float(study_app.best_value)
    })

for param_name, value in study_txn.best_params.items():
    best_params_records.append({
        "model"      : "XGBoost",
        "dataset"    : "transactions",
        "parameter"  : param_name,
        "value"      : float(value),
        "best_pr_auc": float(study_txn.best_value)
    })

best_params_df = pd.DataFrame(best_params_records)

spark.createDataFrame(best_params_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.best_hyperparameters")

print("\nBest hyperparameters saved to Delta")

# Visualization

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Applications history
ax = axes[0, 0]
ax.plot([t.value for t in study_app.trials], marker="o", color="#2563EB")
ax.axhline(study_app.best_value, color="red", linestyle="--",
           label=f"Best: {study_app.best_value:.4f}")
ax.set_title("Optuna Optimization — Applications XGBoost", fontsize=12)
ax.set_xlabel("Trial")
ax.set_ylabel("PR-AUC")
ax.legend()
ax.grid(alpha=0.3)

# Transactions history
ax = axes[0, 1]
ax.plot([t.value for t in study_txn.trials], marker="o", color="#DC2626")
ax.axhline(study_txn.best_value, color="red", linestyle="--",
           label=f"Best: {study_txn.best_value:.4f}")
ax.set_title("Optuna Optimization — Transactions XGBoost", fontsize=12)
ax.set_xlabel("Trial")
ax.set_ylabel("PR-AUC")
ax.legend()
ax.grid(alpha=0.3)

# Parameter importance — applications
ax = axes[1, 0]
importance_app = optuna.importance.get_param_importances(study_app)
ax.barh(list(importance_app.keys()), list(importance_app.values()),
        color="#2563EB")
ax.set_title("Parameter Importance — Applications", fontsize=12)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

# Parameter importance — transactions
ax = axes[1, 1]
importance_txn = optuna.importance.get_param_importances(study_txn)
ax.barh(list(importance_txn.keys()), list(importance_txn.values()),
        color="#DC2626")
ax.set_title("Parameter Importance — Transactions", fontsize=12)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/optuna_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nOptuna analysis complete — check MLflow experiment for full history")